# 🚗 Road Accident Analysis — Data Cleaning & Preparation

### UK Road Safety Collision Data — 2025

**Project Stage:** Data Cleaning & Preparation  
**Dataset:** `collisions_2025.csv`  
**Source:** UK Department for Transport (DfT)

---

## Purpose of This Notebook

The purpose of this notebook is to prepare the raw road collision dataset
for Exploratory Data Analysis.

The cleaning process will focus on:

- Creating a working copy of the raw dataset
- Removing genuine duplicate records if identified
- Validating data types
- Handling missing geographic values appropriately
- Converting date and time fields
- Creating useful time-based features
- Decoding important categorical variables
- Checking invalid or unexpected values
- Investigating potential outliers
- Validating the cleaned dataset

> The original raw dataset will not be modified.

### Import Libraries

In [94]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

Libraries imported successfully!


### Load raw data

In [95]:
# Load the original raw dataset

df_raw = pd.read_csv("../01_Raw_Data/collisions_2025.csv")

print("Raw dataset loaded successfully!")

Raw dataset loaded successfully!


### Create working copy

In [96]:
# Create a working copy for data cleaning

df = df_raw.copy()
print("Working copy created successfully!")

Working copy created successfully!


### Initial validation

In [97]:
# Check the working dataset

print("Rows    :", df.shape[0])
print("Columns :", df.shape[1])

Rows    : 101525
Columns : 44


In [98]:
# Check for complete duplicate records

duplicate_rows = df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

Duplicate rows: 0


### Identifier Validation

In [99]:
# Check collision identifier uniqueness

print("Total records:", len(df))
print("Unique collision_index:", df["collision_index"].nunique())
print("Unique collision_ref_no:", df["collision_ref_no"].nunique())

Total records: 101525
Unique collision_index: 101525
Unique collision_ref_no: 101525


In [100]:
# Check for missing identifier values

print("Missing collision_index:",
      df["collision_index"].isna().sum())

print("Missing collision_ref_no:",
      df["collision_ref_no"].isna().sum())

Missing collision_index: 0
Missing collision_ref_no: 0


### Data type

In [101]:
# Review current data types
df.dtypes

collision_index                                         str
collision_year                                        int64
collision_ref_no                                        str
location_easting_osgr                               float64
location_northing_osgr                              float64
longitude                                           float64
latitude                                            float64
police_force                                          int64
collision_severity                                    int64
number_of_vehicles                                    int64
number_of_casualties                                  int64
date                                                    str
day_of_week                                           int64
time                                                    str
local_authority_district                              int64
local_authority_ons_district                            str
local_authority_highway                 

In [102]:
# Convert date column to datetime

df["date"] = pd.to_datetime(
    df["date"],
    format="%d/%m/%Y",
    errors="coerce"
)

print("Date column converted successfully!")
print("Date data type:", df["date"].dtype)

Date column converted successfully!
Date data type: datetime64[us]


In [103]:
# Check whether any dates became missing after conversion

invalid_dates = df["date"].isna().sum()

print("Invalid or missing dates after conversion:", invalid_dates)

Invalid or missing dates after conversion: 0


### Time cleaning

In [104]:
# Convert time values to a datetime-like time representation

df["time_clean"] = pd.to_datetime(
    df["time"],
    format="%H:%M",
    errors="coerce"
).dt.time

print("Time conversion completed.")

Time conversion completed.


In [105]:
# Check invalid time values

invalid_times = df["time_clean"].isna().sum()
print("Invalid time values:", invalid_times)

Invalid time values: 0


### Create time features

In [106]:
# Extract year from date

df["year"] = df["date"].dt.year

In [107]:
# Extract month number

df["month"] = df["date"].dt.month

In [108]:
# Extract month name

df["month_name"] = df["date"].dt.month_name()

In [109]:
# Extract day of month

df["day"] = df["date"].dt.day

In [110]:
# Extract hour from time

df["hour"] = pd.to_datetime(
    df["time"],
    format="%H:%M",
    errors="coerce"
).dt.hour

In [111]:
# Create time-of-day categories

def classify_part_of_day(hour):
    if pd.isna(hour):
        return np.nan
    elif 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

df["part_of_day"] = df["hour"].apply(classify_part_of_day)

In [112]:
# Create weekday/weekend classification

df["day_type"] = np.where(
    df["day_of_week"].isin([1, 7]),
    "Weekend",
    "Weekday"
)

### Handle Missing Geographic Values

In [113]:
# Check missing geographic values

geo_columns = [
    "location_easting_osgr",
    "location_northing_osgr",
    "longitude",
    "latitude"
]

df[geo_columns].isna().sum()

location_easting_osgr     2
location_northing_osgr    2
longitude                 2
latitude                  2
dtype: int64

In [114]:
# Create a flag indicating whether geographic coordinates are available

df["location_available"] = np.where(
    df["latitude"].notna() & df["longitude"].notna(),
    "Available",
    "Missing"
)

df["location_available"].value_counts()

location_available
Available    101523
Missing           2
Name: count, dtype: int64

### Decode collision severity

In [115]:
# Decode collision severity

severity_map = {
    1: "Fatal",
    2: "Serious",
    3: "Slight"
}

df["collision_severity_label"] = (
    df["collision_severity"]
    .map(severity_map)
)

df["collision_severity_label"].value_counts(dropna=False)

collision_severity_label
Slight     74881
Serious    25191
Fatal       1453
Name: count, dtype: int64

### Decode Day of Week

In [116]:
# Decode day of week

day_map = {
    1: "Sunday",
    2: "Monday",
    3: "Tuesday",
    4: "Wednesday",
    5: "Thursday",
    6: "Friday",
    7: "Saturday"
}

df["day_name"] = df["day_of_week"].map(day_map)

df["day_name"].value_counts()

day_name
Friday       16811
Wednesday    15369
Thursday     15304
Tuesday      14880
Monday       14309
Saturday     13655
Sunday       11197
Name: count, dtype: int64

### Decode road type

In [117]:
# Decode road type

road_type_map = {
    1: "Roundabout",
    2: "One way street",
    3: "Dual carriageway",
    6: "Single carriageway",
    7: "Slip road",
    9: "Unknown"
}

df["road_type_label"] = df["road_type"].map(road_type_map)

df["road_type_label"].value_counts(dropna=False)


road_type_label
Single carriageway    74150
Dual carriageway      14610
Roundabout             6795
One way street         2398
Unknown                1953
Slip road              1619
Name: count, dtype: int64

### Decode Light conditions

In [118]:
# Decode light conditions

light_map = {
    1: "Daylight",
    4: "Darkness - lights lit",
    5: "Darkness - lights unlit",
    6: "Darkness - no lighting",
    7: "Darkness - lighting unknown",
    -1: "Unknown"
}

df["light_conditions_label"] = (
    df["light_conditions"]
    .map(light_map)
)

df["light_conditions_label"].value_counts(dropna=False)

light_conditions_label
Daylight                       72814
Darkness - lights lit          20705
Darkness - no lighting          5682
Darkness - lighting unknown     1588
Darkness - lights unlit          720
Unknown                           16
Name: count, dtype: int64

### Decode weather conditions

In [119]:
# Decode weather conditions

weather_map = {
    1: "Fine - no high winds",
    2: "Raining - no high winds",
    3: "Snowing - no high winds",
    4: "Fine - high winds",
    5: "Raining - high winds",
    6: "Snowing - high winds",
    7: "Fog or mist",
    8: "Other",
    9: "Unknown"
}

df["weather_conditions_label"] = (
    df["weather_conditions"]
    .map(weather_map)
)

df["weather_conditions_label"].value_counts(dropna=False)

weather_conditions_label
Fine - no high winds       83405
Raining - no high winds    10230
Other                       3314
Unknown                     2385
Raining - high winds         880
Fine - high winds            715
Fog or mist                  342
Snowing - no high winds      235
Snowing - high winds          19
Name: count, dtype: int64

### Decode road surface

In [120]:
# Decode road surface conditions

surface_map = {
    1: "Dry",
    2: "Wet or damp",
    3: "Snow",
    4: "Frost or ice",
    5: "Flood over 3cm deep",
    9: "Unknown",
    -1: "Unknown"
}

df["road_surface_conditions_label"] = (
    df["road_surface_conditions"]
    .map(surface_map)
)

df["road_surface_conditions_label"].value_counts(dropna=False)

road_surface_conditions_label
Dry                    75679
Wet or damp            22312
Unknown                 1976
Frost or ice            1261
Flood over 3cm deep      151
Snow                     146
Name: count, dtype: int64

### Decode urban/rural

In [121]:
# Decode urban or rural area

urban_rural_map = {
    1: "Urban",
    2: "Rural",
    3: "Unknown"
}

df["urban_rural_label"] = (
    df["urban_or_rural_area"]
    .map(urban_rural_map)
)

df["urban_rural_label"].value_counts(dropna=False)

urban_rural_label
Urban      67148
Rural      34373
Unknown        4
Name: count, dtype: int64

### Check unmapped values

In [122]:
# Check whether any collision severity values were not mapped

unmapped_severity = (
    df.loc[
        df["collision_severity_label"].isna(),
        "collision_severity"
    ]
    .unique()
)

print("Unmapped severity codes:", unmapped_severity)

Unmapped severity codes: []


In [123]:
# Check unmapped road type codes

unmapped_road_type = (
    df.loc[
        df["road_type_label"].isna(),
        "road_type"
    ]
    .unique()
)

print("Unmapped road type codes:", unmapped_road_type)

Unmapped road type codes: []


In [124]:
# Check unmapped weather codes

unmapped_weather = (
    df.loc[
        df["weather_conditions_label"].isna(),
        "weather_conditions"
    ]
    .unique()
)

print("Unmapped weather codes:", unmapped_weather)

Unmapped weather codes: []


### Investigate casualty outliers

In [125]:
# Examine the distribution of number of casualties

df["number_of_casualties"].describe()

count    101525.000000
mean          1.259621
std           0.816679
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         142.000000
Name: number_of_casualties, dtype: float64

In [126]:
# Display records with the highest reported casualty counts

df[
    [
        "collision_index",
        "date",
        "number_of_vehicles",
        "number_of_casualties",
        "collision_severity_label"
    ]
].sort_values(
    by="number_of_casualties",
    ascending=False
).head(10)

,collision_index,date,number_of_vehicles,number_of_casualties,collision_severity_label
26379,2025052501204,2025-05-26,1,142,Serious
66133,2025041681268,2025-12-03,1,32,Serious
42426,2025520501079,2025-07-17,1,25,Fatal
38141,2025351616661,2025-04-16,3,22,Serious
95613,2025440281666,2025-06-26,2,20,Serious
53804,2025340NN2152,2025-05-25,2,20,Serious
15372,2025161545000,2025-01-22,2,18,Serious
9438,2025430230402,2025-05-11,1,18,Fatal
53839,2025061619139,2025-07-21,1,17,Serious
69882,2025991611220,2025-07-03,2,15,Fatal


### Vehicle count validation

In [127]:
# Check number of vehicles

print(df["number_of_vehicles"].describe())
print("\nMinimum vehicles:", df["number_of_vehicles"].min())
print("Maximum vehicles:", df["number_of_vehicles"].max())

count    101525.000000
mean          1.811849
std           0.689631
min           1.000000
25%           1.000000
50%           2.000000
75%           2.000000
max          17.000000
Name: number_of_vehicles, dtype: float64

Minimum vehicles: 1
Maximum vehicles: 17


In [128]:
# Identify records with unusually high vehicle counts

df[
    df["number_of_vehicles"] >= 10
][
    [
        "collision_index",
        "number_of_vehicles",
        "number_of_casualties",
        "collision_severity_label"
    ]
].sort_values(
    by="number_of_vehicles",
    ascending=False
)

,collision_index,number_of_vehicles,number_of_casualties,collision_severity_label
75831,2025010580108,17,1,Slight
41098,2025211539974,12,2,Serious
15007,2025070860840,12,3,Slight
86141,2025461614378,12,4,Slight
29791,2025471543561,11,1,Serious
60274,2025430064297,11,4,Slight
2206,2025201541593,10,1,Serious
13738,2025451676290,10,2,Serious
37391,2025411600379,10,1,Serious
39509,2025231686575,10,2,Slight


### Check speed limit

In [129]:
# Check speed limit values after cleaning

df["speed_limit"].value_counts().sort_index()

speed_limit
20    20045
30    49673
40     8886
50     4612
60    12709
70     5600
Name: count, dtype: int64

### Check categorical cleaning

In [130]:
# Display cleaned categorical variables

cleaned_category_columns = [
    "collision_severity_label",
    "day_name",
    "road_type_label",
    "light_conditions_label",
    "weather_conditions_label",
    "road_surface_conditions_label",
    "urban_rural_label"
]

df[cleaned_category_columns].head()

,collision_severity_label,day_name,road_type_label,light_conditions_label,weather_conditions_label,road_surface_conditions_label,urban_rural_label
0,Serious,Wednesday,Dual carriageway,Daylight,Fine - no high winds,Dry,Rural
1,Slight,Friday,One way street,Darkness - lights lit,Raining - high winds,Wet or damp,Urban
2,Serious,Saturday,Single carriageway,Darkness - lighting unknown,Unknown,Unknown,Urban
3,Serious,Tuesday,Single carriageway,Darkness - lights lit,Fine - no high winds,Wet or damp,Urban
4,Slight,Thursday,Single carriageway,Daylight,Fine - no high winds,Dry,Urban


### Check new features

In [131]:
# Display newly created analytical features

new_features = [
    "year",
    "month",
    "month_name",
    "day",
    "hour",
    "part_of_day",
    "day_type",
    "location_available"
]

df[new_features].head(10)

,year,month,month_name,day,hour,part_of_day,day_type,location_available
0,2025,3,March,5,12,Afternoon,Weekday,Available
1,2025,10,October,3,18,Evening,Weekday,Available
2,2025,11,November,15,21,Night,Weekend,Available
3,2025,12,December,23,1,Night,Weekday,Available
4,2025,2,February,6,8,Morning,Weekday,Available
5,2025,3,March,5,13,Afternoon,Weekday,Available
6,2025,6,June,28,14,Afternoon,Weekend,Available
7,2025,8,August,19,7,Morning,Weekday,Available
8,2025,9,September,2,9,Morning,Weekday,Available
9,2025,8,August,3,20,Evening,Weekend,Available


### Final data quality check

In [132]:
# Check missing values after cleaning

cleaning_missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().mean() * 100
    ).round(4)
})

cleaning_missing_summary[
    cleaning_missing_summary["Missing_Count"] > 0
].sort_values(
    by="Missing_Count",
    ascending=False
)

,Missing_Count,Missing_Percentage
location_easting_osgr,2,0.002
location_northing_osgr,2,0.002
longitude,2,0.002
latitude,2,0.002


### Final duplicate check

In [133]:
# Verify duplicate rows after preparation

print(
    "Duplicate rows after cleaning:",
    df.duplicated().sum()
)

Duplicate rows after cleaning: 0


### Final Dataset Shape

In [134]:
# Check final dataset dimensions

print("Final number of rows    :", df.shape[0])
print("Final number of columns :", df.shape[1])

Final number of rows    : 101525
Final number of columns : 60


### Data type check

In [135]:
# Review final data types

df.dtypes

collision_index                                                str
collision_year                                               int64
collision_ref_no                                               str
location_easting_osgr                                      float64
location_northing_osgr                                     float64
longitude                                                  float64
latitude                                                   float64
police_force                                                 int64
collision_severity                                           int64
number_of_vehicles                                           int64
number_of_casualties                                         int64
date                                                datetime64[us]
day_of_week                                                  int64
time                                                           str
local_authority_district                                     i

### Final dataset preview

In [136]:
# Display the prepared dataset

df.head()

,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,...,part_of_day,day_type,location_available,collision_severity_label,day_name,road_type_label,light_conditions_label,weather_conditions_label,road_surface_conditions_label,urban_rural_label
0,202517H102225,2025,17H102225,449216.0,534979.0,-1.23770,54.70743,17,2,3,...,Afternoon,Weekday,Available,Serious,Wednesday,Dual carriageway,Daylight,Fine - no high winds,Dry,Rural
1,2025070815222,2025,070815222,363808.0,389192.0,-2.54576,53.39831,7,3,2,...,Evening,Weekday,Available,Slight,Friday,One way street,Darkness - lights lit,Raining - high winds,Wet or damp,Urban
2,2025070937984,2025,070937984,362413.0,389071.0,-2.56673,53.39712,7,2,2,...,Night,Weekend,Available,Serious,Saturday,Single carriageway,Darkness - lighting unknown,Unknown,Unknown,Urban
3,2025041687395,2025,041687395,368238.0,427802.0,-2.48308,53.74562,4,2,1,...,Night,Weekday,Available,Serious,Tuesday,Single carriageway,Darkness - lights lit,Fine - no high winds,Wet or damp,Urban
4,2025161555094,2025,161555094,510792.0,432167.0,-0.32035,53.77406,16,3,2,...,Morning,Weekday,Available,Slight,Thursday,Single carriageway,Daylight,Fine - no high winds,Dry,Urban


### Create clean dataset

In [137]:
# Save the cleaned and prepared dataset

output_path = "../03_Cleaned_Data/road_accidents_2025_cleaned.csv"

df.to_csv(
    output_path,
    index=False
)

print("Cleaned dataset saved successfully!")
print(output_path)

Cleaned dataset saved successfully!
../03_Cleaned_Data/road_accidents_2025_cleaned.csv


### Verify saved file

In [138]:
# Reload the saved dataset to verify the exported file

df_check = pd.read_csv(output_path)

print("Saved dataset verified successfully!")
print(f"Rows    : {df_check.shape[0]:,}")
print(f"Columns : {df_check.shape[1]}")

Saved dataset verified successfully!
Rows    : 101,525
Columns : 60


### Final validation

In [139]:
# Final validation of the cleaned dataset

print("Original rows :", len(df_raw))
print("Cleaned rows  :", len(df_check))

print("\nDuplicate rows:", df_check.duplicated().sum())

print(
    "\nMissing latitude:",
    df_check["latitude"].isna().sum()
)

print(
    "Missing longitude:",
    df_check["longitude"].isna().sum()
)

Original rows : 101525
Cleaned rows  : 101525

Duplicate rows: 0

Missing latitude: 2
Missing longitude: 2


## 11. Data Cleaning Summary

The raw collision dataset was prepared for exploratory analysis while
preserving the original raw data.

### Actions Performed

- Created a working copy of the raw dataset.
- Verified duplicate records and collision identifiers.
- Converted the collision date into a datetime format.
- Validated time values.
- Created year, month, month name, day, and hour features.
- Created time-of-day categories.
- Created weekday/weekend classification.
- Preserved records with missing geographic coordinates.
- Created a geographic availability indicator.
- Decoded important coded categorical variables.
- Validated the decoded categories.
- Investigated unusually high casualty and vehicle counts.
- Performed final duplicate and missing-value checks.
- Exported the prepared dataset for further analysis.

### Data Preservation Principle

No observations were removed solely because they contained unusual values.
Potential outliers were investigated rather than automatically deleted.

The original raw dataset remains unchanged.

## 12. Conclusion

The dataset is now prepared for Exploratory Data Analysis.

The cleaned dataset contains both the original collision variables and
additional analytical features that will make the upcoming analysis easier.

Important derived variables include:

- `year`
- `month`
- `month_name`
- `day`
- `hour`
- `part_of_day`
- `day_type`
- `location_available`

Readable categorical labels have also been created for important variables
such as:

- Collision severity
- Day of week
- Road type
- Light conditions
- Weather conditions
- Road surface conditions
- Urban/rural classification

The prepared dataset has been saved as:

`road_accidents_2025_cleaned.csv`

The next stage of the project is **Exploratory Data Analysis**, where we will
investigate distributions, patterns, relationships, trends, and business
questions using the cleaned dataset.